# AnthropicToolCallState

## Example: Create TOOL_USE state using AnthropicToolUseState

This is actually 2 actions within the same state. The first action is to call the tool and the second action is to pass the result to the LLM and return a response.

```lua
stateDiagram-v2
    INIT --> TOOL_CALL
    TOOL_CALL --> FINAL
```

```mermaid
stateDiagram-v2
    Direction LR
    INIT --> TOOL_CALL
    TOOL_CALL --> FINAL
```


In [1]:
from gai.asm import AsyncStateMachine
from gai.asm import Monologue
from gai.mcp.client import McpAggregatedClient

monologue = Monologue()
monologue.add_user_message("What time is it now?")
monologue.add_assistant_message(
    [
        {
            "citations": None,
            "text": "I'll get the current time for you.",
            "type": "text",
        },
        {
            "id": "toolu_012CyVmXRfMRzxoYBtmoeF1s",
            "input": {"format": "YYYY-MM-DD HH:mm:ss"},
            "name": "current_time",
            "type": "tool_use",
        },
    ]
)

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> TOOL_USE
    TOOL_USE--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                    "mcp_client": {"type": "getter", "dependency": "get_mcp_client"},
                }
            },
            "TOOL_USE": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicToolUseState",
                "title": "TOOL_USE",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                    "mcp_client": {"type": "state_bag", "dependency": "mcp_client"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            # "model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
            "tools": True,
        },
        get_mcp_client=lambda state: McpAggregatedClient(["mcp-time"]),
        monologue=monologue,
    )

## Step 2: INIT --> TOOL_CALL

await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if (isinstance(chunk,str)):
        print(chunk, end='', flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

The current time is **June 16, 2025 at 10:48:26 AM UTC**.

If you'd like to know the time in a specific timezone, please let me know which timezone you're interested in!


1750070904.2777271 User > What time is it now?
1750070904.2779763 Assistant > [{'citations': None, 'text': "I'll get the current time for you.", 'type': 'text'}, {'id': 'toolu_012CyVmXRfMRzxoYBtmoeF1s', 'input': {'format': 'YYYY-MM-DD HH:mm:ss'}, 'name': 'current_time', 'type': 'tool_use'}]
1750070906.3662696 User > [{'type': 'tool_result', 'tool_use_id': 'toolu_012CyVmXRfMRzxoYBtmoeF1s', 'content': 'Current UTC time is 2025-06-16 10:48:26, and the time in UTC is 2025-06-16 10:48:26.'}]
1750070909.7755535 Assistant > [{'citations': None, 'text': "The current time is **June 16, 2025 at 10:48:26 AM UTC**.\n\nIf you'd like to know the time in a specific timezone, please let me know which timezone you're interested in!", 'type': 'text'}]


---

## When LLM has completed its task ("task_completed")

After starting the agent, we will continue calling the agent using continue_async until the agent has completed its task.

But how do we know when the agent has completed its task?

One way is to assume that if the agent does not return a tool_use, then it has completed its task. This works for simpler scenarios but does not work when the agent interrupts itself to ask for more information or to clarify something.

The better way is to create a pseudo tool called "completed_task" that the agent can call when it has completed its task. 

To check whether the agent has completed its task:
- Check if the response is a tool_use
- Check if the tool name is "completed_task"

If this condition is true, then "continue_async" will not stream any more responses.

```lua

In [1]:
from gai.asm import AsyncStateMachine
from gai.asm import Monologue
from gai.mcp.client import McpAggregatedClient

monologue = Monologue()
monologue.add_user_message("What time is it now?")
monologue.add_assistant_message(
    [
        {
            "citations": None,
            "text": "## Summary\n\nI have successfully completed ...",
            "type": "text",
        },
        {
            "id": "toolu_012CyVmXRfMRzxoYBtmoeF1s",
            "input": {},
            "name": "task_completed",
            "type": "tool_use",
        },
    ]
)

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> TOOL_USE
    TOOL_USE--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                    "mcp_client": {"type": "getter", "dependency": "get_mcp_client"},
                }
            },
            "TOOL_USE": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicToolUseState",
                "title": "TOOL_USE",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                    "mcp_client": {"type": "state_bag", "dependency": "mcp_client"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            # "model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
            "tools": True,
        },
        get_mcp_client=lambda state: McpAggregatedClient(["mcp-time"]),
        monologue=monologue,
    )    

## Step 2: INIT --> TOOL_CALL

await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if (isinstance(chunk,str)):
        print(chunk, end='', flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )
    
# The last message should only contain the task_completed tool call, not the current time tool call.





1751196544.7569418 User > What time is it now?
1751196544.757144 Assistant > [{'citations': None, 'text': '## Summary\n\nI have successfully completed ...', 'type': 'text'}, {'id': 'toolu_012CyVmXRfMRzxoYBtmoeF1s', 'input': {}, 'name': 'task_completed', 'type': 'tool_use'}]


---

## When LLM has interrupted itself ("user_input")

When the LLM interrupts itself, it will call a tool and expect a response from the user.
If the user "continue_async", then the agent will not stream any responses.
Instead, the user will provide a "user_message" with continue_async.



In this scenario, the LLM will interrupt the user with a question and expects a response from the user.

The previous message is an agent message that contains the tool "user_message' and the text "Are you asking for the local time or the UTC time?".

#### i) Answer correctly

- Add "Use SGT" to the user message
- Run it.




In [1]:
from gai.asm import AsyncStateMachine
from gai.asm import Monologue
from gai.mcp.client import McpAggregatedClient

monologue = Monologue()
monologue.add_user_message("What time is it now?")
monologue.add_assistant_message(
    [
        {
            "citations": None,
            "text": "Are you asking for the local time or the UTC time?",
            "type": "text",
        },
        {
            "id": "toolu_012CyVmXRfMRzxoYBtmoeF1s",
            "name": "user_input",
            "input": {},
            "type": "tool_use",
        },
    ]
)

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> TOOL_USE
    TOOL_USE--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                    "mcp_client": {"type": "getter", "dependency": "get_mcp_client"},
                }
            },
            "TOOL_USE": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicToolUseState",
                "title": "TOOL_USE",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                    "mcp_client": {"type": "state_bag", "dependency": "mcp_client"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            # "model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
            "tools": True,
        },
        get_mcp_client=lambda state: McpAggregatedClient(["mcp-time"]),
        monologue=monologue,
    )

## Step 2: INIT --> TOOL_CALL

fsm.user_message = "Use SGT"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if isinstance(chunk, str):
        print(chunk, end="", flush=True)
print("\n\n")

## Step 4: Print the state history

for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

# The last message should only contain the task_completed tool call, not the current time tool call.





1751196948.9820013 User > What time is it now?
1751196948.9822693 Assistant > [{'citations': None, 'text': 'Are you asking for the local time or the UTC time?', 'type': 'text'}, {'id': 'toolu_012CyVmXRfMRzxoYBtmoeF1s', 'name': 'user_input', 'input': {}, 'type': 'tool_use'}]
1751196951.079606 User > [{'type': 'tool_result', 'tool_use_id': 'toolu_012CyVmXRfMRzxoYBtmoeF1s', 'content': 'Use SGT'}]
1751196953.6060412 Assistant > [{'id': 'toolu_01B3FUX1fdHQEyqbG2CoeEAv', 'input': {'format': 'YYYY-MM-DD HH:mm:ss', 'timezone': 'Asia/Singapore'}, 'name': 'current_time', 'type': 'tool_use'}]


#### ii) No answer given

- Do not provide user_message.
- Run it.



In [2]:
from gai.asm import AsyncStateMachine
from gai.asm import Monologue
from gai.mcp.client import McpAggregatedClient

monologue = Monologue()
monologue.add_user_message("What time is it now?")
monologue.add_assistant_message(
    [
        {
            "citations": None,
            "text": "Are you asking for the local time or the UTC time?",
            "type": "text",
        },
        {
            "id": "toolu_012CyVmXRfMRzxoYBtmoeF1s",
            "name": "user_input",
            "input": {},
            "type": "tool_use",
        },
    ]
)

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> TOOL_USE
    TOOL_USE--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                    "mcp_client": {"type": "getter", "dependency": "get_mcp_client"},
                }
            },
            "TOOL_USE": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicToolUseState",
                "title": "TOOL_USE",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                    "mcp_client": {"type": "state_bag", "dependency": "mcp_client"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            # "model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
            "tools": True,
        },
        get_mcp_client=lambda state: McpAggregatedClient(["mcp-time"]),
        monologue=monologue,
    )

## Step 2: INIT --> TOOL_CALL

await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if isinstance(chunk, str):
        print(chunk, end="", flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

# Notice that the monologue remains unchanged.





1751197114.0408456 User > What time is it now?
1751197114.0411067 Assistant > [{'citations': None, 'text': 'Are you asking for the local time or the UTC time?', 'type': 'text'}, {'id': 'toolu_012CyVmXRfMRzxoYBtmoeF1s', 'name': 'user_input', 'input': {}, 'type': 'tool_use'}]
